In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

c:\RAG_AGENTIC\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader(r"C:\Users\ipand\Documents\New folder\Download\programming-pytorch-for-deep-learning-creating-and-deploying-deep-learning-applications.pdf").load()
content_page = loader[4:]
print(len(loader))


270


In [3]:
%pip install -U langchain

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
print(len(loader))
print(len(content_page))

270
266


In [5]:
txt_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size = 1024,
    chunk_overlap = 200,
    
)

doc_split = txt_splitter.split_documents(content_page)
print(f"Number of Splits: {len(doc_split)}")
print(f"Sample Chunk: {doc_split[1].page_content}")



Number of Splits: 293
Sample Chunk: matrix	operations,	and	so	these	add-on	graphics	cards	could	be	used	to	speed	up
training	as	well	as	make	larger,	
deeper
	neural	network	architectures	feasible	for
the	first	time.	Other	important	techniques	such	as
	
Dropout
	(which	we	will	look
at	in	
Chapter	3
)	were	also	introduced	in	the	last	decade	as	ways	to	not	just	speed
up	training	but	make	training	more	
generalized
	(so	that	the	network	doesn’t	just
learn	to	recognize	the	training	data,	a	problem	called	
overfitting
	that	we’ll
encounter	in	the	next	chapter).	In	the	last	couple	of	years,	companies	have	taken
this	GPU-based	approach	to	the	next	level,	with	Google	creating	what	it
describes	as
	
tensor	processing	units
	(TPUs),	which	are	devices	custom-built	for
performing	deep	learning	as	fast	as	possible,	and	are	even	available	to	the
general	public	as	part	of	their	Google	Cloud	ecosystem.
Another	way	to	chart	deep	learning’s	progress	over	the	past	decade	is	through
the	ImageNet	competitio

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size =1024,
    chunk_overlap = 200,
    
)

splits = recursive_splitter.split_documents(content_page)
print(len(splits))
print(splits[0].page_content)

581
Preface
Deep	Learning	in	the	World	Today
Hello	and	welcome!	This	book	will	introduce	you	to	deep	learning	via	PyTorch,
an	open	source	library	released	by	
Facebook	in	2017.	Unless	you’ve	had	your
head	stuck	in	the	ground	in	a	very	good	impression	of	an	ostrich	the	past	few
years,	you	can’t	have	helped	but	notice	that	neural	networks	are	everywhere
these	days.	They’ve	gone	from	being	the	
really	cool	bit	of	computer	science	that
people	learn	about	and	then	do	nothing	with
	to	being	carried	around	with	us	in
our	phones	every	day	to	improve	our	pictures	or	listen	to	our	voice	commands.
Our	email	software	reads	our	email	and	produces	context-sensitive	replies,	our
speakers	listen	out	for	us,	cars	drive	by	themselves,	and	the	computer	has	finally
bested	humans	at	Go.	We’re	also	seeing	the	technology	being	used	for	more
nefarious	ends	in	authoritarian	countries,	where	neural	network–backed	sentinels
can	pick	faces	out	of	crowds	and	make	a	decision	on	whether	they	should	be
apprehended.


Parent child chunking :

In [7]:
from langchain_community.retrievers import ParentDocumentRetriever
from langchain_core.stores import InMemoryStore
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000)


child_splitter = RecursiveCharacterTextSplitter(chunk_size=400)


vectorstore = Chroma(
    collection_name="split_parents", 
    embedding_function=HuggingFaceEmbeddings() # Runs locally on your GTX 1650
)
# Store holds the PARENT chunks (The full context)
store = InMemoryStore()

# 4. The Retriever
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

# 5. Add Documents
# The retriever automatically splits into parents AND children, then links them.
retriever.add_documents(content_page)

# 6. Usage
# Returns the LARGE parent chunk, even though it matched the small child
relevant_docs = retriever.invoke("specific query about a detail")

ImportError: cannot import name 'ParentDocumentRetriever' from 'langchain_community.retrievers' (c:\RAG_AGENTIC\.venv\Lib\site-packages\langchain_community\retrievers\__init__.py)

In [ ]:
%pip install --upgrade langchain langchain-community

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# 1. Setup the Embedding Model (Runs locally on your GTX 1650)
print("Loading embedding model...")
embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={'device': 'cuda'}  # <--- USES YOUR GPU
)

# 2. Create the Vector Database
# We verify if 'splits' exists from the previous step
if 'splits' in locals() and len(splits) > 0:
    print(f"Indexing {len(splits)} chunks into ChromaDB...")
    
    vectorstore = Chroma.from_documents(
        documents=splits,                 # The chunks from Step 2
        embedding=embedding_model,        # The model to convert them
        collection_name="agentic_rag",    # Name of your collection
        persist_directory="./chroma_db"   # Saves to disk so you don't lose it
    )
    print("✅ Indexing Complete! Database saved to ./chroma_db")
else:
    print("❌ Error: No 'splits' found. Please run the Chunking step first.")

# 3. Create the Retriever Interface
# This is the tool our Agent will actually "call" later
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3} # Retrieve the top 3 most relevant chunks
)

ConfigError: unable to infer type for attribute "chroma_server_nofile"